# 02 · ERA5-Land Weather Extraction

> **Prerequisite:** `00_setup.ipynb`

In [ ]:
import sys
if '/content/geo_sp_project/src' not in sys.path:
    sys.path.extend(['/content/geo_sp_project/src', '/content/geo_sp_project/configs'])
    import config; from geo_sp.auth import init_ee; init_ee(config.EE_PROJECT)


In [ ]:
import config
from geo_sp.boundaries import get_lga
from geo_sp.era5_weather import (
    get_era5_collection, add_derived_bands,
    extract_timeseries, weather_summary, build_weather_map
)

randwick   = get_lga(config.LGA_NAME, config.LGA_LAYER)
collection = get_era5_collection(randwick, days_back=config.WEATHER_DAYS_BACK)

count = collection.size().getInfo()
print(f'Found {count} hourly images')


In [ ]:
# Compute derived bands + extract time series
derived = collection.map(add_derived_bands)
df      = extract_timeseries(derived, randwick, scale=5000)
print(f'Extracted {len(df)} hourly records')
weather_summary(df)


In [ ]:
# Save CSV to Google Drive
from datetime import datetime
filename = f'randwick_era5_{datetime.now().strftime("%Y%m%d")}.csv'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    df.to_csv(f'/content/drive/MyDrive/{filename}', index=False)
    print(f'Saved to Google Drive: {filename}')
except Exception:
    df.to_csv(filename, index=False)
    print(f'Saved locally: {filename}')


In [ ]:
m = build_weather_map(derived, randwick, config.MAP_CENTER, config.MAP_ZOOM)
m
